# R2 — Refresher: Functions and Dictionaries

**Why this notebook exists:** notebook 03 jumped too fast — straight from one example to a five-part boss fight. This notebook rebuilds the same material more slowly, one concept at a time, with smaller exercises in between. After this you'll go back to notebook 03's E3–E5 and finish the boss fight.

**The pattern:** generic Python example first (just numbers and names) → then the same idea on trading data. You only ever fight one new thing at a time.

**How to use:** predict before running. Type your own code in the empty cells. If a cell errors, read the error — Python errors are usually pretty literal about what went wrong.

---
## 1. What a function actually is

A function is a named block of code you can run later by calling its name.

Think of it like a recipe card. The card itself doesn't cook anything — it just sits in the drawer. You only get food when you take it out and *follow* it. Defining a function is writing the card. Calling the function is following the card.

Three pieces of syntax:

```python
def add(a, b):         # def + name + (arguments) + colon
    result = a + b     # body — indented, like inside a for-loop
    return result      # what to hand back to the caller
```

- `def` is the keyword that means "I'm defining a function."
- `add` is the name. You'll use this to call it later.
- `(a, b)` are the **arguments** — placeholders for values the caller will supply.
- The indented lines are the **body**. They only run when the function is called.
- `return result` ends the function and hands `result` back.

In [ ]:
# Defining the function. This cell does NOT run the body — it only registers the recipe.
def add(a, b):
    result = a + b
    return result

In [ ]:
# Now CALL it. Each call runs the body once with those specific values.
x = add(2, 3)
y = add(10, 7)
print("x is", x)
print("y is", y)

**Key points to lock in:**

1. Running the `def ...` cell does *not* run `a + b`. It only stores the recipe.
2. The values you pass in (`2`, `3`) become `a` and `b` *inside the function only*.
3. After `return`, the function ends. Anything after a `return` on the same indentation level never runs.
4. The result comes back to wherever you called the function — that's what `x = add(2, 3)` captures.

### Exercise 1 — your first function

Write a function called `double` that takes one argument `x` and returns `x * 2`.

Then call it twice — once with `5` and once with `12` — and print the results.

Predict the outputs before you run it.

In [ ]:
def double(x):
    result = x * 2
    return result

y = double(5)
b = double(12)
print("y is" , y)
print("b is" , b)


---
## 2. `print` is not `return` (the most common confusion)

Inside a notebook these look similar because both display values on screen. They are doing very different things.

- `print(x)` *shows* x on the screen. That's it. The caller gets nothing back.
- `return x` *hands x back* to the caller. Nothing is shown unless the caller chooses to print it.

If a function only `print`s and never `return`s, then whoever called it gets back `None` — Python's word for "nothing."

In [ ]:
def add_with_print(a, b):
    print(a + b)        # shows it, but doesn't hand it back

def add_with_return(a, b):
    return a + b        # hands it back

x = add_with_print(2, 3)
y = add_with_return(2, 3)

print("x captured:", x)   # x is None — add_with_print didn't return anything
print("y captured:", y)   # y is 5

**Rule of thumb:** if you want to *use* a value later (store it, do maths with it, pass it to another function), the function must `return` it. `print` is only for showing things on screen.

### Exercise 2 — fix the broken function

Below is a broken version of `square`. It prints the answer but doesn't return it, so the variable `result` ends up being `None`.

Rewrite it so it `return`s the squared value instead of printing it. Then call it and print the captured result.

In [ ]:
def square(x):
   
    val = x * x
    
    return val

result = square(5)


print("captured:", result)

---
## 3. Loops inside functions — and the indentation trap

You already know accumulator loops from notebook 02:

```python
total = 0
for n in numbers:
    total += n
```

Putting that inside a function is just a matter of wrapping it:

```python
def sum_list(numbers):
    total = 0
    for n in numbers:
        total += n
    return total
```

**The trap that bit you in notebook 03:** where you put `return` matters enormously.

Compare these two — the only difference is the indentation of `return`:

In [ ]:
# CORRECT — return is OUTSIDE the loop (same indent as the for line)
def sum_list_correct(numbers):
    total = 0
    for n in numbers:
        total += n
    return total         # <-- runs ONCE, after the loop has finished

print("correct:", sum_list_correct([10, 20, 30, 40]))

In [ ]:
# BROKEN — return is INSIDE the loop (extra indentation)
def sum_list_broken(numbers):
    total = 0
    for n in numbers:
        total += n
        return total     # <-- runs on the FIRST iteration and exits immediately

print("broken:", sum_list_broken([10, 20, 30, 40]))

**What's happening, step by step, in the broken version:**

- Loop iteration 1: `n = 10`. `total` becomes `10`. Then `return total` fires → function exits and hands back `10`.
- Iterations 2, 3, 4 never happen. The loop was interrupted.

**The mental model:** `return` is like an emergency exit. The first time Python hits one, the function is over. If you want the loop to finish first, the `return` must sit *outside* the loop body.

**How to spot it:** look at the indentation. Anything indented under `for ...:` runs once per iteration. The `return` belongs back at the function's top level — same indent as the `for` itself, not as the body of the `for`.

### Exercise 3 — `sum_list` (generic)

Write your own version of `sum_list(numbers)` that returns the sum of all numbers in the list. Use an accumulator loop (no `sum()`).

Test on `[1, 2, 3, 4, 5]` — should give `15`.

Then test on `[]` (empty list) — should give `0`. Confirm both work.

In [ ]:
from numpy import test


def sum_list(numbers):
    total = 0
    for n in numbers:
        total += n


    return total    

result1 = sum_list([1, 2, 3, 4, 5])
print("test [1, 2, 3, 4, 5]" , result1)


### Exercise 4 — `compute_pf(pnls)` (quant)

Now the same shape, but on trading data. `pnls` is a list of trade profits/losses. **Profit factor** = sum of winning trades ÷ absolute value of sum of losing trades. It tells you how many dollars of profit you made per dollar of loss — a profit factor of 2 means you made twice as much on winners as you lost on losers.

Write `compute_pf(pnls)` that returns the profit factor as a float.

**One loop, two trackers** (`gross_profit` and `gross_loss`). The `return` must come *after* the loop finishes — this is the bug that bit you in notebook 03's E1.

Test on `[100, -50, 200, -150, 80]`. Predict the answer first:
- Winners: 100 + 200 + 80 = 380
- Losers: -50 + -150 = -200, abs → 200
- PF = 380 / 200 = ?

In [ ]:
def compute_pf(pnls):
    gross_profit = 0
    gross_loss = 0
    for pnl in pnls:
        if pnl > 0:
            gross_profit += pnl
        elif pnl < 0:
            gross_loss += pnl 
    
    return gross_profit / abs(gross_loss)

    

    
result1 = compute_pf([100, -50, 200, -150, 80])
print("final pf" , result1)


---
## 4. Default arguments

You can give an argument a default value. If the caller doesn't supply one, the default is used.

```python
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}"

greet("Alfie")                    # → "Hello, Alfie"  (uses default greeting)
greet("Alfie", greeting="Hey")    # → "Hey, Alfie"    (overrides it)
```

**Why this is everywhere in the codebase:** the backtester takes ~15 parameters and most have sensible defaults. You only specify the ones that differ from the standard. Without defaults, every call would be 15 arguments long. With defaults, most calls are 2 or 3.

*(That `f"..."` thing is an **f-string** — a string template where anything inside `{ }` gets replaced with the variable's value. You'll see this constantly.)*

In [ ]:
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}"

print(greet("Alfie"))
print(greet("Alfie", greeting="Hey"))

### Exercise 5 — `count_above(numbers, threshold=0)`

Write a function that counts how many numbers in the list are *strictly greater than* `threshold`. Default the threshold to `0`.

Test on `[150, -200, 80, 250, -50, 110]`:
- `count_above(nums)` — count of numbers > 0. Predict.
- `count_above(nums, threshold=100)` — count of numbers > 100. Predict.

**Quant reading of this:** with `threshold=0` this is just "how many winning trades." With `threshold=100` it's "how many trades made more than \$100." Same function, different question.

In [ ]:
def count_above(numbers, threshold = 0):
   count = 0
   
   for n in numbers:
        if n > threshold:
           count += 1

   return count 
    
    
result1 = count_above([150, -200, 80, 250, 110])
result2 = count_above([150, -200, 80, 250, 110], threshold=100)
print("result1" , result1)
print("result2" , result2)



---
## 5. Dictionaries — what they are

A dict is a **labelled lookup**. Like a list, but instead of looking things up by position (`my_list[0]`) you look them up by name (`my_dict["price"]`).

```python
person = {
    "name": "Alfie",
    "age": 30,
    "city": "London",
}
```

- Curly braces `{ }` (not square brackets — those are lists).
- Each entry is `key: value`, comma-separated.
- Keys are usually strings.
- Look up a value with `person["name"]`.

**Why dicts exist:** when several values belong together but mean different things, a list is wrong (positions are meaningless) and separate variables are messy. A dict bundles them with names attached.

In [ ]:
person = {
    "name": "Alfie",
    "age": 30,
    "city": "London",
}

print(person["name"])
print(person["age"])
print("number of entries:", len(person))

In [ ]:
# Add or change entries by assignment:
person["job"] = "researcher"      # adds a new key
person["age"] = 31                # overwrites existing
print(person)

### Exercise 6 — build a dict (generic)

Build a dict called `book` with these keys and values:
- `title` = `"The Man Who Solved the Market"`
- `author` = `"Gregory Zuckerman"`
- `year` = `2019`
- `pages` = `384`

Then:
1. Print just the title using `[ ]` access.
2. Add a new key `"finished"` set to `False`.
3. Print the whole dict.

In [ ]:
book = {
    "title": "The Man Who Solved the Market",
    "author": "Gregory Zuckerman",
    "year": 2019,
    "pages": 384
}

print( book ["title"])
book["finished"] = "False"
print(book)

---
## 6. Looping over a dict — `.items()`

Most of the time when you loop over a dict, you want both the key and the value for each entry. That's what `.items()` gives you:

```python
for key, value in person.items():
    print(key, "->", value)
```

Two variables in the `for` line because each item is a pair. `key` gets the name, `value` gets the value. Run through the loop once per entry in the dict.

In [ ]:
for key, value in person.items():
    print(f"{key}: {value}")

### Exercise 7 — loop over a config (quant)

Here's a real-shaped config dict — a simplified version of `DEFAULT_GATE1_WRAPPER` from `engine/gate1.py`. It bundles the rules for how a Gate 1 backtest executes trades.

```python
wrapper = {
    "stop_atr": 1.5,         # stop-loss at 1.5 x ATR
    "tp_atr": 2.0,           # take-profit at 2 x ATR
    "costs_rt": 5.0,         # round-trip cost per trade ($5)
    "eod_exit": True,        # close any open position at session end
}
```

Loop over it with `.items()` and print each entry as `key: value`. You don't need to understand what every value *means* in trading terms — just print the structure.

In [ ]:

 
wrapper = {
    "stop_atr": 1.5,
    "tp_atr": 2.0,
    "cost_rt": 5.0,
    "eod_exit": True,
}

for key, value in wrapper.items():
    print(f"{key} : {value}")



---
## 7. Functions that return dicts

A function doesn't have to return one number. It can return a whole dict bundling several measurements.

```python
def summarise(numbers):
    return {
        "count": len(numbers),
        "total": sum(numbers),
        "average": sum(numbers) / len(numbers),
    }

result = summarise([10, 20, 30])
print(result["total"])       # 60
print(result["average"])     # 20.0
```

This is the single most common shape in `engine/metrics.py`. The function walks the data once, computes several things, returns one dict with all of them. The caller picks the keys they care about.

In [ ]:
def summarise(numbers):
    return {
        "count": len(numbers),
        "total": sum(numbers),
        "average": sum(numbers) / len(numbers),
    }

result = summarise([10, 20, 30])
print(result)
print("just the total:", result["total"])

### Exercise 8 — `summarise_numbers(numbers)` (generic, combined)

Write a function `summarise_numbers(numbers)` that returns a dict with three keys:
- `count` — how many numbers
- `total` — their sum
- `n_positive` — how many of them are greater than 0

**Constraint:** use *one* for-loop with multiple trackers — don't call `sum()` or `len()`. The point is to combine multiple accumulators in one pass through the list.

Test on `[10, -5, 20, -8, 15]`. Predict each key's value before running:
- count = ?
- total = ?
- n_positive = ?

**Watch the indentation trap:** the `return` must be *outside* the for-loop. Same indent as the `for`, not as the body of the `for`.

In [ ]:
def summarise_numbers(numbers):
    count = 0
    total = 0
    n_positive = 0
    
    for number in numbers:
        
        count += 1
        total += number  
        if number > 0:
            n_positive += 1
            
    
    return {
        "count": count,
        "total": total,
        "n_positive": n_positive
    }


result = summarise_numbers([10, -5, 20, -8, 15])
print(result)



### Exercise 9 — `summarise_trades(pnls)` (quant version of E8)

Same shape as Exercise 8, but on trading P&L data. Write `summarise_trades(pnls)` that returns a dict with:
- `n_trades` — how many trades
- `total_pnl` — sum of all pnls
- `n_winners` — count of trades with pnl > 0
- `gross_profit` — sum of *only* the winning pnls
- `gross_loss` — `abs()` of sum of *only* the losing pnls

**One for-loop, multiple trackers.** Return the dict at the end.

Test on `[100, -50, 200, -150, 80, -30, 120]`. Predict each key before running.

*This is one step short of the `compute_pf` exercise — once you have `gross_profit` and `gross_loss`, the profit factor is just one division. We're keeping them separate so the function returns the raw building blocks, which is exactly what real `engine/metrics.py` functions do.*

In [ ]:



def summarise_trade_pnls(pnls):
    n_trades = 0
    total_pnl = 0
    n_winners = 0
    gross_profit = 0
    gross_loss = 0
    
    for pnl in pnls:
        if pnl > 0:
            n_trades += 1
            total_pnl += pnl
            n_winners += 1
            gross_profit += pnl
        
        elif pnl < 0:
            n_trades += 1
            total_pnl += pnl
            gross_loss += pnl

    return {
        "n_trades": n_trades,
        "total_pnl": total_pnl,
        "n_winners": n_winners,
        "GP": gross_profit,
        "GL": abs(gross_loss),
    }
        
result = summarise_trade_pnls([100, -50, 200, -150, 80, -30, 120])
print(result)




---
## Done — what's next

If Exercise 9 worked, you've now hit every piece of the notebook 03 boss fight individually:
- Functions with arguments ✅
- Default arguments ✅
- One loop, multiple trackers ✅
- Returning a dict ✅
- Iterating a dict ✅
- `return` placement (outside the loop) ✅

**Tell me how it went:**
1. Where did you slow down or get stuck? (Be specific — "the return-in-loop trap" or "f-strings" or "dict iteration".)
2. Does the difference between `print` and `return` feel automatic now, or still requires conscious thought?
3. Does a dict feel like a natural choice when you want to bundle several related values?

**Then we go back to notebook 03** and finish E3, E4, E5 — they should feel much less scary now. The boss fight (`evaluate_gate1`) is just combining E9's output with a threshold dict and comparing key-by-key. You've built all the pieces.